# mSigLIP Colab Training Experiments

Notebook này gom các bước cần để chạy training experiments trên Google Colab khi `VN3K/` và `m_siglip_checkpoints/` đã có sẵn trên Google Drive.

Workflow chính hiện tại:

1. Chạy **MNEB-HN evidence-only trên VN3K sạch** để xác nhận module mới no-op về training effect khi auxiliary losses tắt.
2. Chạy **MNEB-HN trên CUHK-PEDES natural noise**: evidence bank + FNM auxiliary + RDE auxiliary, trong khi Circle Loss vẫn giữ nguyên hard-negative branch.
3. Chạy **PRW-TPS-CN tiếng Trung** để kiểm tra multilingual generalization.
4. Chạy **LoRA attn+FFN r32/PiSSA** với batch nhỏ + gradient accumulation khi Colab thiếu tài nguyên.

Notebook dùng TensorBoard mặc định, artifact/checkpoint lưu thẳng vào Drive để có thể resume khi Colab bị ngắt.


In [ ]:
!cp "/content/drive/MyDrive/m_siglip_checkpoints.zip" "/content/msiglip_code"

In [ ]:
!mkdir "/content/msiglip_data"

In [ ]:
!cp "/content/drive/MyDrive/VN3K.zip" "/content/msiglip_data"

In [ ]:
!unzip -q "/content/msiglip_data/VN3K.zip" -d "/content/msiglip_data"

In [ ]:
!unzip -q "/content/msiglip_code/m_siglip_checkpoints.zip" -d "/content/msiglip_code"

## 0. Runtime Check

Chạy cell này trước để xác nhận Colab đang dùng GPU runtime. Nếu `torch.cuda.is_available()` là `False`, vào `Runtime > Change runtime type > GPU` rồi chạy lại notebook.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path


def run(cmd, check=True, cwd=None, env=None):
    print(f"$ {cmd}", flush=True)
    return subprocess.run(cmd, shell=True, check=check, cwd=cwd, env=env)


def run_capture(cmd, check=True, cwd=None, env=None):
    print(f"$ {cmd}", flush=True)
    proc = subprocess.run(
        cmd,
        shell=True,
        cwd=cwd,
        env=env,
        text=True,
        capture_output=True,
    )
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)
    if check and proc.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {proc.returncode}: {cmd}")
    return proc


def run_stream(cmd, check=True, cwd=None, env=None):
    print(f"$ {cmd}", flush=True)
    proc_env = os.environ.copy()
    proc_env["PYTHONUNBUFFERED"] = "1"
    proc_env["HYDRA_FULL_ERROR"] = "1"
    proc_env["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
    if env:
        proc_env.update(env)

    proc = subprocess.Popen(
        cmd,
        shell=True,
        cwd=cwd,
        env=proc_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="", flush=True)

    code = proc.wait()
    if check and code != 0:
        raise RuntimeError(f"Command failed with exit code {code}: {cmd}")
    return code


run("nvidia-smi", check=False)

try:
    import torch
    print("Python:", sys.version)
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("CUDA:", torch.version.cuda)
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as exc:
    print("Torch import failed:", repr(exc))


## 1. Mount Drive + Path Config

Sửa các biến dưới đây nếu Drive của bạn đặt repo/data/model ở chỗ khác.

Hai layout phổ biến:

1. Chuẩn hóa:
   - `/content/drive/MyDrive/data/raw/VN3K/`
   - `/content/drive/MyDrive/artifacts/models/pretrained/m_siglip_checkpoints/model.safetensors`
2. Đặt trực tiếp dưới MyDrive:
   - `/content/drive/MyDrive/VN3K/`
   - `/content/drive/MyDrive/m_siglip_checkpoints/model.safetensors`

In [ ]:
from pathlib import Path
import os

DRIVE_CODE_DIR = Path("/content/drive/MyDrive/mSigLIP/code_v2")
LOCAL_CODE_DIR = Path("/content/msiglip_code")
DRIVE_VN3K = DRIVE_CODE_DIR / "VN3K"
LOCAL_DATA_ROOT = Path("/content/msiglip_data/mnt/data/user_data/lampt/PS/code")

DRIVE_PRETRAINED = DRIVE_CODE_DIR / "m_siglip_checkpoints"
LOCAL_PRETRAINED_ROOT = Path("/content/msiglip_code/mnt/data/user_data/lampt/PS/code")

PROJECT_DIR = LOCAL_CODE_DIR
MSIGLIP_DATA_ROOT = str(LOCAL_DATA_ROOT)
MSIGLIP_PRETRAINED_ROOT = str(LOCAL_PRETRAINED_ROOT)

# Artifacts vẫn lưu Drive để không mất khi runtime disconnect.
MSIGLIP_ARTIFACTS_ROOT = "/content/drive/MyDrive/mSigLIP/code/artifacts"

os.environ["MSIGLIP_DATA_ROOT"] = MSIGLIP_DATA_ROOT
os.environ["MSIGLIP_PRETRAINED_ROOT"] = MSIGLIP_PRETRAINED_ROOT
os.environ["MSIGLIP_ARTIFACTS_ROOT"] = MSIGLIP_ARTIFACTS_ROOT

print("PROJECT_DIR =", PROJECT_DIR)
print("MSIGLIP_DATA_ROOT =", MSIGLIP_DATA_ROOT)
print("MSIGLIP_PRETRAINED_ROOT =", MSIGLIP_PRETRAINED_ROOT)
print("MSIGLIP_ARTIFACTS_ROOT =", MSIGLIP_ARTIFACTS_ROOT)

## 2. Repo Setup

Cell này chuyển vào repo và cài package ở chế độ editable. Notebook dùng `pip`, không dùng `uv`, vì Colab đã có Python runtime riêng.

Nếu Colab báo thiếu package, giữ `INSTALL_MINIMAL_DEPS=True`. Nếu dependency đã đầy đủ và muốn nhanh hơn, đổi thành `False`.

In [ ]:
import os
import sys
from pathlib import Path

assert PROJECT_DIR.exists(), f"PROJECT_DIR does not exist: {PROJECT_DIR}"
os.chdir(PROJECT_DIR)
print('cwd =', Path.cwd())

# PROJECT_DIR must be the full mSigLIP repo, not just a folder containing this notebook, VN3K, and checkpoints.
core_repo_files = [
    PROJECT_DIR / 'pyproject.toml',
    PROJECT_DIR / 'trainer.py',
    PROJECT_DIR / 'src' / 'msiglip' / 'train.py',
    PROJECT_DIR / 'configs' / 'cir_msiglip.yaml',
    PROJECT_DIR / 'configs' / 'loss' / 'cir_msiglip.yaml',
    PROJECT_DIR / 'configs' / 'lora' / 'default.yaml',
]
missing_core = [str(path) for path in core_repo_files if not path.exists()]
if missing_core:
    print('PROJECT_DIR is not a complete/current repo. Missing core files:')
    for path in missing_core:
        print(' -', path)
    raise FileNotFoundError(
        'Upload/sync the full repo to Google Drive or set PROJECT_DIR to the real repo folder. '
        'VN3K and m_siglip_checkpoints alone are not enough to run training.'
    )

INSTALL_MINIMAL_DEPS = True

# Make local src importable immediately in this kernel and subprocesses. This is
# enough for this repo because trainer.py also inserts PROJECT_DIR/src itself.
src_dir = str(PROJECT_DIR / 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
os.environ['PYTHONPATH'] = src_dir + os.pathsep + os.environ.get('PYTHONPATH', '')

if INSTALL_MINIMAL_DEPS:
    # Keep torch from the Colab runtime. Install only project-side dependencies commonly missing in Colab.
    run(
        f"{sys.executable} -m pip install -q "
        "hydra-core omegaconf lightning loguru prettytable peft safetensors "
        "transformers sentencepiece ftfy tensorboard wandb scikit-learn scipy seaborn matplotlib nltk"
    )
    # Colab may preinstall an old torchao. PEFT 0.19 refuses torchao<0.16 even for normal LoRA.
    # We do not use torchao here, so uninstalling it is safer than upgrading the Colab Torch stack.
    run(f"{sys.executable} -m pip uninstall -y -q torchao", check=False)


# Editable install is convenient but not required for this notebook. On some
# Colab/Python/setuptools combinations it can fail while direct PYTHONPATH works.
editable = run(f"{sys.executable} -m pip install -e . --no-deps", check=False)
if editable.returncode != 0:
    print("WARNING: editable install failed; continuing with PYTHONPATH fallback:", src_dir)
    print("If you want the exact pip error, rerun: python -m pip install -e . --no-deps -v")

# The text augmentation module initializes NLTK stopwords at import time.
try:
    import nltk
    nltk.download('stopwords', quiet=True)
    nltk.download('wordnet', quiet=True)
    nltk.download('omw-1.4', quiet=True)
except Exception as exc:
    print('NLTK data setup warning:', repr(exc))

print('Setup complete')

## 3. Asset Verification

Fail sớm nếu data/model path sai. Đổi `ASSET_DATASET_CONFIG` sang dataset đang định chạy.

- VN3K: `$MSIGLIP_DATA_ROOT/VN3K` với `data_captions*.json`
- CUHK-PEDES: `$MSIGLIP_DATA_ROOT/CUHK-PEDES` với `reid_raw.json`
- PRW-TPS-CN: `$MSIGLIP_DATA_ROOT/PRW-TPS-CN` với `prw_cn_caption_crops.json`
- `$MSIGLIP_PRETRAINED_ROOT/m_siglip_checkpoints/model.safetensors`

In [ ]:
from pathlib import Path
import os

DATA_ROOT = Path(os.environ['MSIGLIP_DATA_ROOT'])
PRETRAINED_ROOT = Path(os.environ['MSIGLIP_PRETRAINED_ROOT'])
ARTIFACTS_ROOT = Path(os.environ['MSIGLIP_ARTIFACTS_ROOT'])

ASSET_DATASET_CONFIG = 'cuhk_pedes'  # options: vn3k_vi, vn3k_en, vn3k_mixed, cuhk_pedes, cuhk_pedes_10_percent, prw_tps_cn
DATASET_ASSET_SPECS = {
    'vn3k_vi': (DATA_ROOT / 'VN3K', ['data_captions_vn3k.json'], 'imgs'),
    'vn3k_en': (DATA_ROOT / 'VN3K', ['data_captions.json'], 'imgs'),
    'vn3k_mixed': (DATA_ROOT / 'VN3K', ['data_captions.json'], 'imgs'),
    'cuhk_pedes': (DATA_ROOT / 'CUHK-PEDES', ['reid_raw.json'], 'imgs'),
    'cuhk_pedes_10_percent': (DATA_ROOT / 'CUHK-PEDES', ['reid_raw.json'], 'imgs'),
    'prw_tps_cn': (DATA_ROOT / 'PRW-TPS-CN', ['prw_cn_caption_crops.json'], 'imgs'),
}
assert ASSET_DATASET_CONFIG in DATASET_ASSET_SPECS, f"Unknown ASSET_DATASET_CONFIG: {ASSET_DATASET_CONFIG}"

dataset_dir, annotation_names, image_dir_name = DATASET_ASSET_SPECS[ASSET_DATASET_CONFIG]
model_path = PRETRAINED_ROOT / 'm_siglip_checkpoints' / 'model.safetensors'

print('Dataset config:', ASSET_DATASET_CONFIG)
print('Dataset dir:', dataset_dir)
print('Model:', model_path)
print('Artifacts:', ARTIFACTS_ROOT)

assert dataset_dir.exists(), f"Missing dataset dir: {dataset_dir}"
assert model_path.exists(), f"Missing mSigLIP checkpoint: {model_path}"

annotation_files = [dataset_dir / name for name in annotation_names if (dataset_dir / name).exists()]
image_dir = dataset_dir / image_dir_name
image_samples = sorted(image_dir.rglob('*.jpg'))[:5] + sorted(image_dir.rglob('*.png'))[:5]

print('Annotation files:')
for path in annotation_files[:10]:
    print(' -', path)
print('Image samples:')
for path in image_samples[:10]:
    print(' -', path)

assert annotation_files, f"Missing annotation file(s) {annotation_names} under {dataset_dir}"
assert image_dir.exists(), f"Missing image dir: {image_dir}"
assert image_samples, f"No image samples found under {image_dir}"

## 4. Preflight Cho MNEB-HN

Chạy unit tests nhanh cho evidence bank, MNEB objectives, integration guard, và LoRA configs. Không dùng `fast_dev_run` ở đây vì retrieval validation cần đủ image/text positives; `fast_dev_run` cắt validation quá ngắn và có thể tạo lỗi metric giả.


In [ ]:
required_files = [
    PROJECT_DIR / 'src' / 'msiglip' / 'model' / 'evidence_bank.py',
    PROJECT_DIR / 'tests' / 'test_evidence_bank.py',
    PROJECT_DIR / 'tests' / 'test_mneb_objectives.py',
    PROJECT_DIR / 'tests' / 'test_mneb_integration.py',
    PROJECT_DIR / 'configs' / 'loss' / 'cir_msiglip.yaml',
    PROJECT_DIR / 'configs' / 'lora' / 'default.yaml',
    PROJECT_DIR / 'configs' / 'lora' / 'attn_ffn_r32_pissa.yaml',
    PROJECT_DIR / 'configs' / 'lora' / 'attn_ffn_r32_dora.yaml',
    PROJECT_DIR / 'configs' / 'lora' / 'attn_ffn_r32_rslora.yaml',
    PROJECT_DIR / 'run_mneb_hn.sh',
]

missing = [str(path) for path in required_files if not path.exists()]
if missing:
    print('Missing required repo files:')
    for path in missing:
        print(' -', path)
    raise FileNotFoundError('Colab repo is missing recent MNEB-HN files. Re-package and upload the latest training code.')

run_capture(
    f'{sys.executable} -c \'import peft, hydra, omegaconf; '
    'print("peft", peft.__version__); print("hydra ok"); print("omegaconf ok")\''
)
# Use unittest discover instead of tests/test_*.py paths because Colab can import a
# third-party `tests` package before the local tests directory.
run_capture(f"{sys.executable} -m unittest discover -s tests -p 'test_evidence_bank.py' -v")
run_capture(f"{sys.executable} -m unittest discover -s tests -p 'test_mneb_objectives.py' -v")
run_capture(f"{sys.executable} -m unittest discover -s tests -p 'test_mneb_integration.py' -v")
run_capture(f"{sys.executable} -m unittest discover -s tests -p 'test_lora_configs.py' -v")


## 5. MNEB-HN Clean VN3K

Chạy trước để kiểm tra MNEB-HN trên VN3K sạch. Run `mneb_clean_evidence_only` chỉ thu evidence/diagnostics và không thêm training loss; run `mneb_clean_full` bật FNM/RDE auxiliary losses để kiểm tra module mới.

Mặc định MNEB cells dùng recipe clean tốt nhất hiện tại: `+lora=attn_ffn_r32` và `loss.PART_ALIGN=true`. Đây không phải full fine-tune.

Mặc định cell dùng `dataset.batch_size=12`, `trainer.accumulate_grad_batches=11` để phù hợp Colab khi không còn đủ tài nguyên cho batch 64.


In [ ]:
# PEFT trên Colab có thể vấp torchao cũ. LoRA thường không cần torchao.
run_capture(f"{sys.executable} -m pip uninstall -y torchao", check=False)

COLAB_BATCH_SIZE = 12
COLAB_ACCUM = 11

CLEAN_COMMON = [
    f"{sys.executable} -u trainer.py -cn cir_msiglip",
    "trainer.max_epochs=60",
    f"dataset.batch_size={COLAB_BATCH_SIZE}",
    "dataset.test_batch_size=128",
    "dataset.num_workers=4",
    f"trainer.accumulate_grad_batches={COLAB_ACCUM}",
    "++trainer.precision=16-mixed",
    "logger.logger_type=tensorboard",
    "optimizer=cir_test",
    "optimizer.param_groups.default.lr=1e-4",
    "loss.NACIR=false",
    "loss.MNEB=true",
    "loss.mneb_config.evidence_bank.enabled=true",
    "loss.PART_ALIGN=true",
    "dataset.noisy_rate=0.0",
    "dataset.noisy_file=null",
    "dataset.fn_noisy_rate=0.0",
    "dataset.fn_noisy_file=null",
]


def clean_cmd(name, fnm_enabled, rde_enabled, lora="attn_ffn_r32"):
    return " ".join(
        CLEAN_COMMON
        + [
            f"logger.experiment_name={name}",
            f"loss.mneb_config.fnm_aux.enabled={'true' if fnm_enabled else 'false'}",
            f"loss.mneb_config.rde_aux.enabled={'true' if rde_enabled else 'false'}",
            f"+lora={lora}",
        ]
    )

clean_commands = {
    "mneb_clean_evidence_only": clean_cmd("mneb_clean_evidence_only_part_align_r32_b12_acc11", False, False),
    "mneb_clean_full": clean_cmd("mneb_clean_full_part_align_r32_b12_acc11", True, True),
}

# Backward-compatible generic name for resume examples below.
clean_main_cmd = clean_commands["mneb_clean_evidence_only"]

for key, cmd in clean_commands.items():
    print(f"\n# {key}\n{cmd}")


In [ ]:
# Chạy evidence-only trước để xác nhận no-op; đổi sang mneb_clean_full khi muốn bật aux losses.
RUN_NAME = "mneb_clean_evidence_only"
run_stream(clean_commands[RUN_NAME])


## 6. MNEB-HN Trên CUHK-PEDES Natural Noise

Chạy sau khi VN3K clean acceptance ổn. CUHK-PEDES là benchmark chính cho MNEB-HN vì có natural FP/FN/ambiguous-caption noise; không inject synthetic noise vào VN3K trong workflow chính nữa.

- `cuhk_part_align_*` là baseline cùng LoRA/backbone/Part Align nhưng không bật MNEB.
- `cuhk_mneb_*_evidence_only` chỉ thu diagnostics, không thêm auxiliary loss.
- `cuhk_mneb_*_full` bật FNM/RDE auxiliary losses.
- Nếu Colab không đủ tài nguyên cho full CUHK, đổi `CUHK_DATASET_CONFIG` sang `cuhk_pedes_10_percent` để smoke-test pipeline trước.

Yêu cầu dữ liệu: `$MSIGLIP_DATA_ROOT/CUHK-PEDES` với `reid_raw.json` và thư mục `imgs/`.


In [ ]:
from pathlib import Path
import os
import sys

CUHK_DATASET_CONFIG = "cuhk_pedes"  # switch to "cuhk_pedes_10_percent" for a quick smoke test
CUHK_LORA = "attn_ffn_r32"  # keep same LoRA for baseline/MNEB comparison
cuhk_dir = Path(os.environ["MSIGLIP_DATA_ROOT"]) / "CUHK-PEDES"
print("CUHK-PEDES dir:", cuhk_dir)
assert cuhk_dir.exists(), f"Missing CUHK-PEDES dir: {cuhk_dir}"
assert (cuhk_dir / "reid_raw.json").exists(), f"Missing CUHK annotation: {cuhk_dir / 'reid_raw.json'}"

CUHK_COMMON = [
    f"{sys.executable} -u trainer.py -cn cir_msiglip",
    f"dataset={CUHK_DATASET_CONFIG}",
    "trainer.max_epochs=60",
    f"dataset.batch_size={COLAB_BATCH_SIZE}",
    "dataset.test_batch_size=128",
    "dataset.num_workers=4",
    f"trainer.accumulate_grad_batches={COLAB_ACCUM}",
    "++trainer.precision=16-mixed",
    "logger.logger_type=tensorboard",
    "optimizer=cir_test",
    "optimizer.param_groups.default.lr=1e-4",
    "loss.PART_ALIGN=true",
    "dataset.noisy_rate=0.0",
    "dataset.noisy_file=null",
    "dataset.fn_noisy_rate=0.0",
    "dataset.fn_noisy_file=null",
]


def cuhk_cmd(name, mneb, fnm_enabled, rde_enabled, lora=CUHK_LORA):
    mneb_overrides = [
        f"loss.MNEB={'true' if mneb else 'false'}",
        "loss.NACIR=false",
        f"+lora={lora}",
    ]
    if mneb:
        mneb_overrides += [
            "loss.mneb_config.evidence_bank.enabled=true",
            f"loss.mneb_config.fnm_aux.enabled={'true' if fnm_enabled else 'false'}",
            f"loss.mneb_config.rde_aux.enabled={'true' if rde_enabled else 'false'}",
        ]
    return " ".join(
        CUHK_COMMON
        + [
            f"logger.experiment_name={name}",
        ]
        + mneb_overrides
    )

cuhk_commands = {
    "cuhk_part_align_r32": cuhk_cmd("cuhk_part_align_r32_b12_acc11", False, False, False),
    "cuhk_mneb_evidence_only_part_align_r32": cuhk_cmd("cuhk_mneb_evidence_only_part_align_r32_b12_acc11", True, False, False),
    "cuhk_mneb_full_part_align_r32": cuhk_cmd("cuhk_mneb_full_part_align_r32_b12_acc11", True, True, True),
}

CUHK_RUN_ORDER = [
    "cuhk_part_align_r32",
    "cuhk_mneb_evidence_only_part_align_r32",
    "cuhk_mneb_full_part_align_r32",
]

for key in CUHK_RUN_ORDER:
    print(f"\n# {key}\n{cuhk_commands[key]}")


In [ ]:
# Chạy từng experiment một. Đổi RUN_NAME theo CUHK_RUN_ORDER ở cell trên.
RUN_NAME = "cuhk_part_align_r32"
run_stream(cuhk_commands[RUN_NAME])


## 7. PRW-TPS-CN Tiếng Trung

Section này chạy benchmark tiếng Trung. PRW-TPS-CN không phải mục tiêu natural-noise chính như CUHK-PEDES, nhưng là sanity check quan trọng cho multilingual mSigLIP.

- `prw_part_align_r32` là baseline cùng LoRA/backbone/Part Align nhưng không bật MNEB.
- `prw_mneb_evidence_only_part_align_r32` chỉ thu diagnostics MNEB-HN.
- `prw_mneb_full_part_align_r32` bật FNM/RDE auxiliary losses nếu muốn kiểm tra framework trên PRW.

Yêu cầu dữ liệu: `$MSIGLIP_DATA_ROOT/PRW-TPS-CN` với `prw_cn_caption_crops.json` và thư mục `imgs/`.


In [ ]:
from pathlib import Path
import os
import sys

PRW_LORA = "attn_ffn_r32"
prw_dir = Path(os.environ["MSIGLIP_DATA_ROOT"]) / "PRW-TPS-CN"
print("PRW-TPS-CN dir:", prw_dir)
assert prw_dir.exists(), f"Missing PRW-TPS-CN dir: {prw_dir}"
assert (prw_dir / "prw_cn_caption_crops.json").exists(), f"Missing PRW annotation: {prw_dir / 'prw_cn_caption_crops.json'}"
assert (prw_dir / "imgs").exists(), f"Missing PRW image dir: {prw_dir / 'imgs'}"

PRW_COMMON = [
    f"{sys.executable} -u trainer.py -cn cir_msiglip",
    "dataset=prw_tps_cn",
    "trainer.max_epochs=60",
    f"dataset.batch_size={COLAB_BATCH_SIZE}",
    "dataset.test_batch_size=128",
    "dataset.num_workers=4",
    f"trainer.accumulate_grad_batches={COLAB_ACCUM}",
    "++trainer.precision=16-mixed",
    "logger.logger_type=tensorboard",
    "optimizer=cir_test",
    "optimizer.param_groups.default.lr=1e-4",
    "loss.PART_ALIGN=true",
    "dataset.noisy_rate=0.0",
    "dataset.noisy_file=null",
    "dataset.fn_noisy_rate=0.0",
    "dataset.fn_noisy_file=null",
]


def prw_cmd(name, mneb, fnm_enabled, rde_enabled, lora=PRW_LORA):
    mneb_overrides = [
        f"loss.MNEB={'true' if mneb else 'false'}",
        "loss.NACIR=false",
        f"+lora={lora}",
    ]
    if mneb:
        mneb_overrides += [
            "loss.mneb_config.evidence_bank.enabled=true",
            f"loss.mneb_config.fnm_aux.enabled={'true' if fnm_enabled else 'false'}",
            f"loss.mneb_config.rde_aux.enabled={'true' if rde_enabled else 'false'}",
        ]
    return " ".join(
        PRW_COMMON
        + [f"logger.experiment_name={name}"]
        + mneb_overrides
    )

prw_commands = {
    "prw_part_align_r32": prw_cmd("prw_part_align_r32_b12_acc11", False, False, False),
    "prw_mneb_evidence_only_part_align_r32": prw_cmd("prw_mneb_evidence_only_part_align_r32_b12_acc11", True, False, False),
    "prw_mneb_full_part_align_r32": prw_cmd("prw_mneb_full_part_align_r32_b12_acc11", True, True, True),
}

PRW_RUN_ORDER = [
    "prw_part_align_r32",
    "prw_mneb_evidence_only_part_align_r32",
    "prw_mneb_full_part_align_r32",
]

for key in PRW_RUN_ORDER:
    print(f"\n# {key}\n{prw_commands[key]}")


In [ ]:
# Chạy từng experiment một. Đổi RUN_NAME theo PRW_RUN_ORDER ở cell trên.
RUN_NAME = "prw_part_align_r32"
run_stream(prw_commands[RUN_NAME])


## 8. LoRA Variants Để Tăng R@1

Section này dùng cho clean-performance search. Hiện baseline clean tốt nhất trong journal là Circle + LoRA attn+FFN r32 batch64, nhưng khi Colab thiếu tài nguyên thì dùng batch12/accum11 để giữ effective batch gần tương đương.

Mặc định section này chạy `vn3k_vi`. Đổi `LORA_DATASET_CONFIG` trong cell dưới sang `vn3k_en`, `vn3k_mixed`, `cuhk_pedes`, `cuhk_pedes_10_percent`, hoặc `prw_tps_cn` nếu muốn chạy cùng LoRA variants trên dataset khác.

Khuyến nghị thứ tự hiện tại: `circle_attn_ffn_r32_pissa_colab` trước, rồi `circle_part_align_r32_pissa_colab`, `circle_attn_ffn_r32_colab`, `rslora`, hoặc `dora` nếu còn runtime.


In [ ]:
LORA_DATASET_CONFIG = "vn3k_vi"  # options: vn3k_vi, vn3k_en, vn3k_mixed, cuhk_pedes, cuhk_pedes_10_percent, prw_tps_cn
LORA_DATASET_ROOTS = {
    "vn3k_vi": DATA_ROOT / "VN3K",
    "vn3k_en": DATA_ROOT / "VN3K",
    "vn3k_mixed": DATA_ROOT / "VN3K",
    "cuhk_pedes": DATA_ROOT / "CUHK-PEDES",
    "cuhk_pedes_10_percent": DATA_ROOT / "CUHK-PEDES",
    "prw_tps_cn": DATA_ROOT / "PRW-TPS-CN",
}
LORA_REQUIRED_PATHS = {
    "vn3k_vi": ["data_captions_vn3k.json", "imgs"],
    "vn3k_en": ["data_captions.json", "imgs"],
    "vn3k_mixed": ["data_captions.json", "imgs"],
    "cuhk_pedes": ["reid_raw.json", "imgs"],
    "cuhk_pedes_10_percent": ["reid_raw.json", "imgs"],
    "prw_tps_cn": ["prw_cn_caption_crops.json", "imgs"],
}
assert LORA_DATASET_CONFIG in LORA_DATASET_ROOTS, f"Unknown LORA_DATASET_CONFIG: {LORA_DATASET_CONFIG}"

lora_dataset_dir = LORA_DATASET_ROOTS[LORA_DATASET_CONFIG]
print("LoRA dataset config:", LORA_DATASET_CONFIG)
print("LoRA dataset dir:", lora_dataset_dir)
assert lora_dataset_dir.exists(), f"Missing LoRA dataset dir: {lora_dataset_dir}"
for relative_path in LORA_REQUIRED_PATHS[LORA_DATASET_CONFIG]:
    required_path = lora_dataset_dir / relative_path
    assert required_path.exists(), f"Missing required LoRA dataset path: {required_path}"


def lora_experiment_name(name):
    if LORA_DATASET_CONFIG == "vn3k_vi":
        return name
    return f"{name}_{LORA_DATASET_CONFIG}"


STRONG_COMMON = [
    f"{sys.executable} -u trainer.py -cn cir_msiglip",
    f"dataset={LORA_DATASET_CONFIG}",
    "trainer.max_epochs=60",
    f"dataset.batch_size={COLAB_BATCH_SIZE}",
    "dataset.test_batch_size=128",
    "dataset.num_workers=4",
    f"trainer.accumulate_grad_batches={COLAB_ACCUM}",
    "++trainer.precision=16-mixed",
    "logger.logger_type=tensorboard",
    "optimizer=cir_test",
    "optimizer.param_groups.default.lr=1e-4",
    "loss.NACIR=false",
    "loss.MNEB=false",
    "dataset.noisy_rate=0.0",
    "dataset.noisy_file=null",
    "dataset.fn_noisy_rate=0.0",
    "dataset.fn_noisy_file=null",
]


def strong_cmd(name, *extra):
    return " ".join(STRONG_COMMON + [f"logger.experiment_name={lora_experiment_name(name)}"] + list(extra))

strong_commands = {
    "circle_attn_ffn_r32_pissa_colab": strong_cmd(
        "circle_attn_ffn_r32_pissa_b12_acc11", "loss.PART_ALIGN=false", "+lora=attn_ffn_r32_pissa"
    ),
    "circle_attn_ffn_r32_colab": strong_cmd(
        "circle_attn_ffn_r32_b12_acc11", "loss.PART_ALIGN=false", "+lora=attn_ffn_r32"
    ),
    "circle_attn_ffn_r32_rslora_colab": strong_cmd(
        "circle_attn_ffn_r32_rslora_b12_acc11", "loss.PART_ALIGN=false", "+lora=attn_ffn_r32_rslora"
    ),
    "circle_attn_ffn_r32_dora_colab": strong_cmd(
        "circle_attn_ffn_r32_dora_b12_acc11", "loss.PART_ALIGN=false", "+lora=attn_ffn_r32_dora"
    ),
    "circle_attn_ffn_r64_colab": strong_cmd(
        "circle_attn_ffn_r64_b12_acc11", "loss.PART_ALIGN=false", "+lora=attn_ffn_r64"
    ),
    "circle_part_align_r32_pissa_colab": strong_cmd(
        "circle_part_align_r32_pissa_b12_acc11", "loss.PART_ALIGN=true", "+lora=attn_ffn_r32_pissa"
    ),
    "circle_part_align_r32_colab": strong_cmd(
        "circle_part_align_r32_b12_acc11", "loss.PART_ALIGN=true", "+lora=attn_ffn_r32"
    ),
}

for key, cmd in strong_commands.items():
    print(f"\n# {key}\n{cmd}")


In [ ]:
# Chạy từng experiment một. PiSSA là candidate đang ưu tiên khi tài nguyên Colab hạn chế.
RUN_NAME = "circle_attn_ffn_r32_pissa_colab"
run_stream(strong_commands[RUN_NAME])


## 9. Resume + Monitor

Dùng section này nếu Colab disconnect hoặc hết phiên. Checkpoint `last.ckpt` được lưu dưới Drive artifacts. Chọn đúng base command đang chạy rồi append `+ckpt_path=...`.

`run_stream()` đã set `TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1` để resume checkpoint Lightning trên PyTorch 2.6+.


In [ ]:
from pathlib import Path
import os

runs_dir = Path(os.environ['MSIGLIP_ARTIFACTS_ROOT']) / 'training' / 'runs'
last_ckpts = sorted(
    runs_dir.glob('**/last.ckpt'),
    key=lambda p: p.stat().st_mtime if p.exists() else 0,
)

if last_ckpts:
    latest_ckpt = last_ckpts[-1]
    print('Latest last.ckpt:', latest_ckpt)
    print('\nRecent checkpoints:')
    for path in last_ckpts[-10:]:
        print(path)
else:
    latest_ckpt = None
    print('No last.ckpt found under', runs_dir)


def resume_cmd(base_cmd, ckpt_path=None):
    ckpt = ckpt_path or latest_ckpt
    if ckpt is None:
        raise FileNotFoundError(f'No last.ckpt found under {runs_dir}')
    return f"{base_cmd} +ckpt_path={ckpt}"


In [ ]:
# Ví dụ resume clean MNEB evidence-only. Đổi base command nếu đang resume robustness/LoRA run.
if latest_ckpt is not None:
    resume_clean_mneb_cmd = resume_cmd(clean_commands["mneb_clean_evidence_only"])
    print(resume_clean_mneb_cmd)
    # run_stream(resume_clean_mneb_cmd)


In [ ]:
# TensorBoard. If the magic has trouble with env vars, paste the printed path manually.
print('TensorBoard logdir:', Path(os.environ['MSIGLIP_ARTIFACTS_ROOT']) / 'training' / 'runs')


In [ ]:
%load_ext tensorboard
%tensorboard --logdir "$MSIGLIP_ARTIFACTS_ROOT/training/runs"


## 10. Result Collection

Cell này liệt kê log/config/checkpoint quan trọng để tải về hoặc ghi vào journal sau full run.


In [ ]:
from pathlib import Path
import os

root = Path(os.environ['MSIGLIP_ARTIFACTS_ROOT']) / 'training' / 'runs'
patterns = ['**/train.log', '**/.hydra/config.yaml', '**/checkpoints/*.ckpt', '**/events.out.tfevents.*']

for pattern in patterns:
    matches = sorted(root.glob(pattern), key=lambda p: p.stat().st_mtime if p.exists() else 0)
    print(f"\n# {pattern} ({len(matches)} files)")
    for path in matches[-20:]:
        print(path)

print('\nSau khi có full-run result, ghi metric vào docs/journal/[train]-YYYY-MM-DD.md theo policy repo.')
